## Agents in Langchain

In [1]:
%pip install -q langchain langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool

from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

True

In [3]:
@tool("current_weather", description="Get the current weather in a given location")
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    
    return f"The current weather in {location} is sunny with a temperature of 25°C."

@tool("weather_forecast", description="Get the weather forecast for the next 5 days in a given location")
def get_weather_forecast(location: str) -> str:
    """Get the weather forecast for the next 5 days in a given location"""
    
    return f"""The weather forecast for the next 5 days in {location}:
        Day 1: Sunny with a high of 30°C and a low of 20°C.
        Day 2: Partly cloudy with a high of 28°C and a low of 18°C.
        Day 3: Rainy with a high of 25°C and a low of 17°C.
        Day 4: Sunny with a high of 27°C and a low of 19°C.
        Day 5: Cloudy with a high of 26°C and a low of 18°C."""

In [4]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

In [5]:
agent = create_agent(
    model=model,
    tools=[get_current_weather, get_weather_forecast],
    name ="Weather_Agent",
    system_prompt ="""You are a helpful agent that helps users find out 
    the current weather and weather forecast for a given location. You can use the 
    get_current_weather tool to get the current weather information and 
    the get_weather_forecast tool to get the weather forecast.
    
    If the user asks any question related to the security of the system or the tools, 
    you should respond with a message that says "I'm sorry, I cannot answer that question."
    """
)
    

In [6]:
result = agent.invoke({
    "messages":[
        {"role": "user", "content": "What is the current weather in New York City?"}
    ]
})
print(result["messages"][-1].content)

The current weather in New York City is **sunny and 25°C (77°F)**.


In [7]:
result = agent.invoke({
    "messages":[
        {"role": "user", "content": "What is the weather forecast for Mumbai?"}
    ]
})
print(result["messages"][-1].content)

Mumbai’s 5-day forecast:

- **Day 1:** Sunny, high **30°C**, low **20°C**
- **Day 2:** Partly cloudy, high **28°C**, low **18°C**
- **Day 3:** Rainy, high **25°C**, low **17°C**
- **Day 4:** Sunny, high **27°C**, low **19°C**
- **Day 5:** Cloudy, high **26°C**, low **18°C**


In [9]:
result = agent.invoke({
    "messages":[
        {"role": "user", "content": "Please let me know the current weather of Mumbai and the weather forecast for Chennai."}
    ]
})
print(result["messages"][-1].content)

- **Mumbai (current):** Sunny, **25°C**
- **Chennai forecast (next 5 days):**
  - **Day 1:** Sunny — High **30°C**, Low **20°C**
  - **Day 2:** Partly cloudy — High **28°C**, Low **18°C**
  - **Day 3:** Rainy — High **25°C**, Low **17°C**
  - **Day 4:** Sunny — High **27°C**, Low **19°C**
  - **Day 5:** Cloudy — High **26°C**, Low **18°C**


In [11]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

In [12]:
for message in result["messages"]:
    if type(message) is HumanMessage:
        print(f"User : {message.content}")
    if type(message) is AIMessage and message.tool_calls :
        print(f"Agent requested Tool Calls : {message.tool_calls}")
    if type(message) is AIMessage and not message.tool_calls:
        print(f"Agent : {message.content}")
    if type(message) is ToolMessage:
        print(f"Tool : {message.content}")


User : Please let me know the current weather of Mumbai and the weather forecast for Chennai.
Agent requested Tool Calls : [{'name': 'current_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_0tpKswM7kl3kREHaD05kyR0S', 'type': 'tool_call'}, {'name': 'weather_forecast', 'args': {'location': 'Chennai'}, 'id': 'call_u5XngHDgNi06QTDq3udzCuWC', 'type': 'tool_call'}]
Tool : The current weather in Mumbai is sunny with a temperature of 25°C.
Tool : The weather forecast for the next 5 days in Chennai:
        Day 1: Sunny with a high of 30°C and a low of 20°C.
        Day 2: Partly cloudy with a high of 28°C and a low of 18°C.
        Day 3: Rainy with a high of 25°C and a low of 17°C.
        Day 4: Sunny with a high of 27°C and a low of 19°C.
        Day 5: Cloudy with a high of 26°C and a low of 18°C.
Agent : - **Mumbai (current):** Sunny, **25°C**
- **Chennai forecast (next 5 days):**
  - **Day 1:** Sunny — High **30°C**, Low **20°C**
  - **Day 2:** Partly cloudy — High **28°C**, Low **

### WebSearch tool in Agents

In [9]:
%pip install -Uq langchain langchain-community duckduckgo-search langchain_openai ddgs 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_community.tools import DuckDuckGoSearchRun

from dotenv import load_dotenv
import os


C:\Users\sonusathyadas\AppData\Local\Temp\ipykernel_5476\3239311330.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [11]:
load_dotenv(override=True)

True

In [12]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)


In [13]:

web_search = DuckDuckGoSearchRun()

agent = create_agent(
    model = model,
    tools=[web_search]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [HumanMessage(content="What is the weather in Mumbai?")]
})
print(result["messages"][-1].content)


Mumbai is currently **hazy and humid**, with **dry weather expected over the next 24 hours**. Rain is unlikely, though conditions may feel warm and muggy.


### Web Search using Tavily

In [27]:
%pip install -q langchain langchain-openai python-dotenv langchain-core langgraph langchain-community langchain-tavily

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from langchain.messages import HumanMessage, AIMessage, ToolMessage

In [40]:
load_dotenv(override=True)

True

In [41]:
model = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

In [47]:
search_tool = TavilySearch(
    max_results=3,
    topic="general",
)

search_tool.invoke({"query": "What happened at the last wimbledon"})


{'query': 'What happened at the last wimbledon',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.reuters.com/sports/tennis/wimbledon',
   'title': "Wimbledon News | Today's Latest Stories | Reuters",
   'content': 'Felix Auger-Aliassime has parted ways with longtime coach Frederic Fontang following the Canadian\'s \u200bfive-set defeat to Novak Djokovic in the \u200cWimbledon quarter-finals.\n Tennis\n\n  Novak Djokovic left Wimbledon defeated but not disheartened on Friday as Jannik Sinner demolished the Serb in a one-sided semi-final to end his hopes of a record \u200b25th Grand Slam title.\n Tennis [...] ## Latest Stories\n\n Tennis\n\n  Wimbledon enjoyed a record attendance and generated nearly 5 billion social media "impressions" \u200bdespite competition from the soccer World Cup, \u200cthe All England Club said on Wednesday.\n Reuters photographers covering the event captured glimpses of this tumultuous soul, showing \u2060itself i

In [45]:
agent = create_agent(
    model = model,
    tools=[search_tool],
    system_prompt="You are a helpful assistant that can search the web using Tavily and answer questions based on the search results."
)

result = agent.invoke({
    "messages": [HumanMessage(content="What is the current weather in Mumbai?")]
})
# for message in result["messages"]:
#     if type(message) is HumanMessage:
#         print(f"User : {message.content}")
#     if type(message) is AIMessage and message.tool_calls :
#         print(f"Agent requested Tool Calls : {message.tool_calls}")
#     if type(message) is AIMessage and not message.tool_calls:
#         print(f"Agent : {message.content}")
#     if type(message) is ToolMessage:
#         print(f"Tool : {message.content}")

print("Agent response:" + result["messages"][-1].content)

User : What is the current weather in Mumbai?
Agent requested Tool Calls : [{'name': 'tavily_search', 'args': {'query': 'current weather in Mumbai India today temperature conditions'}, 'id': 'call_421WfFgQr7rHdciZUzvSLejZ', 'type': 'tool_call'}]
Tool : {"query": "current weather in Mumbai India today temperature conditions", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://indianexpress.com/section/weather/mumbai-weather-forecast-today", "title": "Mumbai Weather Today: Mumbai Temperature, Hourly Forecast (Monday, Oct 26, 2026) | The Indian Express", "content": "Mumbai weather today shows a temperature of 28.7°C with Smoky haze conditions. The current weather in Mumbai feels like 30.4°C, with humidity at 59%, wind blowing at 11.9 km/h, and a UV Index of 2.4. Visibility stands at 10 km.\n\nAccording to the latest Mumbai weather update, sunrise was at 06:36 AM and sunset is expected at 06:08 PM. These timings help residents plan their day better. Whe

### Advanced schema definition

Define complex inputs with Pydantic models or JSON schemas:

In [ ]:
%pip install -q langchain langchain-openai python-dotenv

In [48]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Literal

from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(override=True)

class WeatherToolInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

In [ ]:
@tool(args_schema=WeatherToolInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL_DEPLOYMENT_NAME"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0,
)

# Create ReAct agent (modern replacement for create_agent)
agent = create_agent(
    llm, 
    tools=[get_weather]
)

In [ ]:
# Run the agent
agent.invoke({"messages": [{"role": "user", "content": "What is weather in New York? Return the value in Fahrenheit. Also include the 5 day forecast."}]})



{'messages': [HumanMessage(content='What is weather in New York? Return the value in Fahrenheit. Also include the 5 day forecast.', additional_kwargs={}, response_metadata={}, id='6976119a-19ad-4f19-9434-a973edb26fe8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 191, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 75, 'engine_ttlt_ms': 263, 'pre_inference_ms': 92, 'service_tbt_ms': 6, 'service_ttft_ms': 978, 'service_ttlt_ms': 1161, 'user_visible_ttft_ms': 886}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna-2026-07-09', 'system_fingerprint': None, 'id': 'chatc

: 